# Evaluating translations

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/byu-matrix-lab/torchlingo/blob/main/docs/docs/tutorials/07-evaluating-translations.ipynb)

Every other tutorial asks *can the model translate?* This one asks the question you actually get paid for: **given two systems, which one ships?**

That is a decision, not a number, and the metric you reach for changes the answer. By the end of this notebook you will have seen the default metric pick the worse of two systems, and you will know how to catch that.

**This notebook needs no model, no corpus and no GPU.** Every example below is a handful of fixed strings. That is deliberate: a real model varies in a dozen ways at once, and you cannot learn what a metric responds to while everything moves. Fixed strings let each example change exactly one thing. It also means the whole notebook runs in a free Colab runtime, with nothing to download.

The companion prose page, [Evaluation](../concepts/evaluation.md), covers what the metrics *are*. This notebook is about what they do to you.

<!-- GENERATED:purpose - change the torchlingo metadata, not this cell -->

**TorchLingo tutorial** — assigned reading for Lecture 6 (*Automatic and Human MT Evaluation*). A head start on **A6** (due at Lecture 7), **A8** (due at Lecture 9).

## Part 1: Two systems, one decision

Here are four reference translations and two systems that tried to produce them.

In [1]:
from torchlingo.evaluation import compute_bleu, compute_chrf, compute_ter

REFERENCES = [
    "The committee approved the proposal yesterday",
    "She walked quickly to the station",
    "They are building a new library downtown",
    "He forgot his umbrella at the office",
]

SYSTEM_A = [
    "The committee approved yesterday the proposal",
    "She walked to the station quickly",
    "They are building downtown a new library",
    "He forgot at the office his umbrella",
]

SYSTEM_B = [
    "The committee rejected the proposal yesterday",
    "She walked slowly to the station",
    "They are building a new museum downtown",
    "He forgot his wallet at the office",
]

for i, ref in enumerate(REFERENCES):
    print(f"reference  {ref}")
    print(f"system A   {SYSTEM_A[i]}")
    print(f"system B   {SYSTEM_B[i]}")
    print()

reference  The committee approved the proposal yesterday
system A   The committee approved yesterday the proposal
system B   The committee rejected the proposal yesterday

reference  She walked quickly to the station
system A   She walked to the station quickly
system B   She walked slowly to the station

reference  They are building a new library downtown
system A   They are building downtown a new library
system B   They are building a new museum downtown

reference  He forgot his umbrella at the office
system A   He forgot at the office his umbrella
system B   He forgot his wallet at the office



**Stop and decide before you score anything.** Which system would you ship?

Read the pairs once more and name the difference between the two systems in words. The whole lesson of this notebook depends on you having an opinion before a number arrives to replace it.

Now score them. Three metrics, all shipped in `torchlingo.evaluation`, all from sacreBLEU.

Note the direction: **TER is an error rate, so lower is better.** The other two are higher-is-better. Mixing those up is the single most common way to read an evaluation table backwards.

In [2]:
def score(system, references):
    """Return the three corpus scores for one system."""
    return {
        "BLEU": compute_bleu(system, references).score,
        "chrF": compute_chrf(system, references).score,
        "TER": compute_ter(system, references).score,
    }


scores = {"A": score(SYSTEM_A, REFERENCES), "B": score(SYSTEM_B, REFERENCES)}

print(f"{'':10s} {'BLEU':>8s} {'chrF':>8s} {'TER':>8s}")
print(f"{'':10s} {'(higher)':>8s} {'(higher)':>8s} {'(lower)':>8s}")
for name, row in scores.items():
    print(f"system {name:3s} {row['BLEU']:8.2f} {row['chrF']:8.2f} {row['TER']:8.2f}")

print()
for metric, better in (("BLEU", max), ("chrF", max), ("TER", min)):
    a, b = scores["A"][metric], scores["B"][metric]
    # A tie is a real answer. Handing max/min a tie makes it return whichever
    # argument came first, which reads as a preference the metric never had.
    if a == b:
        print(f"{metric:5s} cannot separate them, {a:.2f} either way")
    else:
        print(f"{metric:5s} prefers system {'A' if better(a, b) == a else 'B'}")

               BLEU     chrF      TER
           (higher) (higher)  (lower)
system A      28.19    82.65    15.38
system B      41.59    72.02    15.38

BLEU  prefers system B
chrF  prefers system A
TER   cannot separate them, 15.38 either way


### The three metrics do not agree, and that is the point

BLEU prefers **B**. chrF prefers **A**. TER cannot tell them apart at all.

So look at what the two systems actually did, because only one of them changed the meaning:

| System | What it did | Meaning |
|---|---|---|
| A | moved words around: *approved yesterday the proposal* | preserved |
| B | swapped one content word: approved → **rejected**, library → **museum**, umbrella → **wallet** | **destroyed** |

System B says the committee *rejected* the proposal. That is not a clumsy translation, it is the opposite of the truth. System A is awkward and entirely faithful.

**BLEU, the default metric in this library and most papers, picked the worse system.** Not because BLEU is broken, but because of what it counts: BLEU rewards matching word n-grams, and B keeps long runs of the reference intact while corrupting one word inside them. A breaks every 4-gram by reordering, and pays for it, even though a reader loses nothing.

chrF disagrees because it counts character n-grams: *rejected* shares almost nothing with *approved* at the character level, so B's substitutions cost it real credit, while A's reordering leaves the characters mostly where they were.

TER ties because it counts **edits**, and both systems need exactly one edit per sentence: A needs one shift, B needs one substitution. TER charges the same for moving a phrase and for reversing a sentence's meaning.

The lesson is not "use chrF". It is that **a single metric is a single opinion, and a disagreement between metrics is information you would not otherwise have.** When two metrics rank two systems differently, that is the moment to go read the output.

## Part 2: There is no such thing as an average of sentence BLEUs

A natural instinct: score each sentence, then average. It feels more informative than one corpus number, and it lets you find the worst sentence.

It also gives a different answer.

In [3]:
import sacrebleu

per_sentence = [
    sacrebleu.sentence_bleu(hyp, [ref]).score
    for hyp, ref in zip(SYSTEM_A, REFERENCES)
]

for hyp, s in zip(SYSTEM_A, per_sentence):
    print(f"{s:6.2f}  {hyp}")

mean_of_sentences = sum(per_sentence) / len(per_sentence)
corpus = compute_bleu(SYSTEM_A, REFERENCES).score

print()
print(f"mean of sentence BLEUs : {mean_of_sentences:.2f}")
print(f"corpus BLEU            : {corpus:.2f}")
print(f"difference             : {mean_of_sentences - corpus:.2f}")

 39.76  The committee approved yesterday the proposal
 39.76  She walked to the station quickly
 42.73  They are building downtown a new library
 35.93  He forgot at the office his umbrella

mean of sentence BLEUs : 39.55
corpus BLEU            : 28.19
difference             : 11.36


Same translations, same references, and the two numbers are far apart.

Neither is a rounding error of the other, because they are not the same quantity. Corpus BLEU pools the n-gram counts across every sentence and computes **one** precision per n-gram order from those totals. Averaging sentence scores computes many precisions from tiny counts and then averages the results, which is not the same operation and weights short sentences far more heavily.

Two practical consequences:

- **Report corpus scores.** If you report an average of sentence scores, say so explicitly, because it is not comparable to anybody else's BLEU.
- **Sentence BLEU is close to meaningless on short sentences.** `sentence_bleu` applies smoothing to avoid the zeros that plain BLEU would produce, which is why the numbers above are not zero. Without it, a sentence shorter than four words has no 4-grams, the 4-gram precision is zero, and the geometric mean collapses regardless of quality. The [Evaluation](../concepts/evaluation.md) page shows a perfect translation scoring 0.0 for exactly that reason.

That trap is not hypothetical in this repository. A test here once compared two BLEU scores that were **both 0.0**, because the sentences were too short to contain a 4-gram, and passed while measuring nothing at all.

## Part 3: The number is incomplete without its settings

"chrF" is not one metric. It is a family, and which member you got depends on a parameter.

In [4]:
print("all of these are labelled 'chrF', on identical data:")
measured = {}
for word_order in (0, 1, 2, 3):
    measured[word_order] = compute_chrf(
        SYSTEM_A, REFERENCES, word_order=word_order
    ).score
    label = "  <- TorchLingo's default" if word_order == 2 else ""
    print(f"  word_order={word_order}  {measured[word_order]:6.2f}{label}")

print(f"\nspread across the four: {max(measured.values()) - min(measured.values()):.2f} points")

# The same disagreement across two tools, which is how it actually bites:
# compute_chrf defaults to word_order=2, sacreBLEU's corpus_chrf defaults to 0.
short_predictions = ["Hello world", "How are you"]
short_references = ["Hello world", "How are you doing"]

ours = compute_chrf(short_predictions, short_references).score
theirs = sacrebleu.corpus_chrf(short_predictions, [short_references]).score

print("\nsame data, both called 'chrF', two libraries at their own defaults:")
print(f"  compute_chrf            {ours:6.2f}   (word_order=2)")
print(f"  sacrebleu.corpus_chrf   {theirs:6.2f}   (word_order=0)")
print(f"  gap                     {ours - theirs:6.2f}")

all of these are labelled 'chrF', on identical data:
  word_order=0   82.92
  word_order=1   85.36
  word_order=2   82.65  <- TorchLingo's default
  word_order=3   76.55

spread across the four: 8.81 points

same data, both called 'chrF', two libraries at their own defaults:
  compute_chrf             78.40   (word_order=2)
  sacrebleu.corpus_chrf    76.97   (word_order=0)
  gap                       1.42


`word_order=0` is plain chrF. `word_order=2` adds word bigrams and is usually written **chrF++**; it is what `compute_chrf` uses by default. sacreBLEU's own `corpus_chrf` defaults to `0`.

That last pair is the one that bites, because both libraries call their output "chrF" and neither says which. The final block above scores one set of translations with each library at its own default, and the two numbers differ. Put those side by side in a table labelled "chrF" and you have published a comparison of nothing.

This exact mismatch has already produced two wrong conclusions in this project. A correct implementation was judged broken against the wrong baseline, and a bug report stated its true value next to a snippet that computes a different one. In both cases the code was right and the undeclared `word_order` default was the whole error.

The defence is a **signature**: a string recording the settings that produced the score, so the number travels with its own provenance.

In [5]:
result = compute_bleu(SYSTEM_A, REFERENCES)
print(f"BLEU  {result.score:.2f}")
print(f"      {result.signature}")

BLEU  28.19
      nrefs:1|case:mixed|eff:no|tok:13a|smooth:exp|version:2.6.0


`tok:13a` is the part that most often invalidates a comparison: change the tokenizer and the same translations score differently. `nrefs:1` says one reference per sentence. `version` pins sacreBLEU itself, because metric implementations change between releases.

**Report the signature with the score.** A BLEU number without one is not reproducible, and is not comparable to anybody else's.

## Part 4: The shape of your references, and a wrong answer with no error

This is the most useful habit in the notebook, because it is the failure mode that does not announce itself.

sacreBLEU's corpus functions want references as **streams**: one list per reference *slot*, each running the whole length of the corpus. The natural way to hold references is the transpose of that, one list per *sentence*. The two shapes look equally plausible:

```python
# one list per sentence  (natural, and wrong for sacreBLEU)
[["ref 1"], ["ref 2"], ["ref 3"]]

# one list per reference slot  (what sacreBLEU wants)
[["ref 1", "ref 2", "ref 3"]]
```

Pass the first one and sacreBLEU does not raise. It reads three sentences as three separate reference streams holding one sentence each, scores against that, and hands back a number.

In [6]:
predictions = ["Hello world", "How are you"]
references = ["Hello world", "How are you doing"]

# The wrong shape: one list per sentence, handed straight to sacreBLEU.
per_sentence_shape = [[ref] for ref in references]
wrong = sacrebleu.corpus_chrf(predictions, per_sentence_shape, word_order=2).score

# The library's function, which reshapes for you.
right = compute_chrf(predictions, references).score

print(f"wrong reference shape : {wrong:6.2f}   <- no exception, no warning")
print(f"compute_chrf          : {right:6.2f}")

wrong reference shape : 100.00   <- no exception, no warning
compute_chrf          :  78.40


A perfect score, silently, on a corpus where one translation is missing a word.

The mechanism: with one reference stream per sentence, every sentence is compared against a *set* of references that happens to contain the right one for it, so everything matches something. The metric is not lying. It is answering a different question than the one you asked.

This was a real bug in `torchlingo.evaluation`. `compute_chrf` and `compute_ter` passed the per-sentence shape through for their whole existence, and nothing caught it, because no test and no tutorial ever called them. `compute_bleu` happened to transpose correctly, which is why BLEU numbers in this repo were fine and the other two were not.

All three now route through one helper, so the reshaping cannot be right in one place and wrong in another:

In [7]:
from torchlingo.evaluation import _as_reference_streams

print("one reference per sentence:")
print(" ", _as_reference_streams(["ref 1", "ref 2", "ref 3"]))

print("\ntwo references per sentence, transposed into two streams:")
print(" ", _as_reference_streams([["a1", "a2"], ["b1", "b2"], ["c1", "c2"]]))

print("\nragged input is refused rather than silently truncated:")
try:
    _as_reference_streams([["a1", "a2"], ["b1"]])
except ValueError as err:
    print(" ", err)

one reference per sentence:
  [['ref 1', 'ref 2', 'ref 3']]

two references per sentence, transposed into two streams:
  [['a1', 'b1', 'c1'], ['a2', 'b2', 'c2']]

ragged input is refused rather than silently truncated:
  every sentence must carry the same number of references; got [1, 2]


**The habit worth taking from this part:** when you adopt a metric, score a case whose answer you already know. Two identical strings should give you a perfect score, and two unrelated strings should give you a terrible one. If either end is surprising, the shape of your inputs is wrong, and no exception is coming to tell you.

## Part 5: What none of these numbers can tell you

All three metrics compare against a reference, which means they can only ask *how close is this to one particular good translation*. They cannot ask:

- **Is it even the target language?** A model that copies its input scores non-zero against a related-language reference, and looks weak rather than broken.
- **Is it adequate, or merely fluent?** Part 1 is exactly this. System B was fluent and wrong; BLEU preferred it.
- **Is the reference any good?** A single reference is one translator's choice among many valid ones, and every metric here treats it as the truth.

[Evaluation](../concepts/evaluation.md) covers what to do about that, including the metric families that need no reference at all, and what counts as a good score in the first place.

## Your turn

**1. Build a system that beats both.** Write a `SYSTEM_C` that preserves meaning *and* word order, score it, and confirm all three metrics agree for once. Agreement is the easy case, which is why it teaches less than Part 1.

**2. Make TER disagree.** Part 1 left TER tied. Change one sentence of `SYSTEM_A` or `SYSTEM_B` so TER breaks the tie, and say which system it now prefers and why. Remember it counts edits, and lower is better.

**3. Reproduce the trap on BLEU.** Part 4 used chrF. Try the same wrong reference shape with `sacrebleu.corpus_bleu` and note what happens. Then explain why `compute_bleu` was unaffected by the original bug.

**4. Find the sentence that is dragging a score down.** Corpus scores hide their worst case. Using per-sentence scores to *locate* a problem is legitimate; using their average to *report* quality is not. Write a few lines that print the sentences sorted worst-first, and say which of those two things you are doing.

## Summary

- **A metric is an opinion.** BLEU preferred the system that reversed the meaning of every sentence, because it counts matching word n-grams and B kept those intact.
- **Disagreement between metrics is a signal**, not noise. It is the cheapest indication that you should go read the output.
- **Corpus scores and averaged sentence scores are different quantities.** Report corpus scores, and say so if you do anything else.
- **A score without its settings is not comparable.** `word_order` alone moved chrF several points under one label. Report the signature.
- **Verify a metric on a case whose answer you know**, because the wrong reference shape returns a plausible number rather than an error.

## What's next

[Evaluation](../concepts/evaluation.md) for score bands, metric families and the reference-free metrics. [Diagnosing failures](06-diagnosing-failures.ipynb) for the checks that run before you trust any number at all, including the one that asks whether your measurement is lying.